# Compound AI Systems Exercise — Assessment 04

**Role context:** Researcher interview, live notebook prototyping  
**Time limit: 45 minutes. No AI coding assistance.**


## Problem Statement

A strategy team asks for a **short research brief** on whether to pilot MoveBot robots. You are given heterogeneous source notes. Your system must **synthesize a recommendation** while **separating supported claims from contested or unsupported ones**.

This is research synthesis with evidence discipline — not generic summarization.


## Functional Requirements

**Input**
- `query: str` — research question / decision ask
- `sources: list[dict]` with `doc_id`, `title`, `text`

**Output**
```python
{
  "executive_brief": "150-250 words, decision-focused",
  "recommendation": "pilot" | "do_not_pilot" | "needs_more_data",
  "recommendation_rationale": "...",
  "claims": [
    {
      "claim": "...",
      "status": "supported" | "contested" | "unsupported" | "unclear",
      "evidence_doc_ids": ["D1", "D3"],
      "notes": "how evidence maps to the claim"
    }
  ],
  "risks": ["..."],
  "open_questions": ["..."],
  "source_coverage": {"used": ["D1", "D3"], "unused": ["D5"]}
}
```

**Constraints**
- Every factual claim in the brief should appear in `claims` with status.
- Vendor marketing vs independent review must be distinguishable via `contested` / notes.
- Do not invent statistics absent from sources.
- Unused sources should appear in `source_coverage.unused` with intent (either truly irrelevant or an oversight you accept).

**Edge cases**
- Contradictory vendor vs third-party numbers.
- Safety industry-wide stats that are not vendor-specific.
- Missing ROI math inputs.

**Desired behavior**
A VP should see what is known, what is disputed, and what to ask next.


## Available Tools & Starter Code


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
SOURCES = [
  {
    "doc_id": "D1",
    "title": "Q2 Market Note \u2014 Warehouse Robotics",
    "text": "Global warehouse robotics deployments grew an estimated 18% YoY in Q2. Adoption is strongest in North America e-commerce fulfillment. Average payback period cited by mid-market buyers: 22 months. Labor shortage remains the primary buying trigger."
  },
  {
    "doc_id": "D2",
    "title": "Vendor Brief \u2014 MoveBot 3000",
    "text": "MoveBot claims 35% pick-rate improvement versus manual baselines in three pilot sites. Pilots were 8\u201312 weeks. Two of three sites were apparel; one grocery. List price starts at $48k per robot excluding integration. MoveBot has 120 production deployments globally per company blog."
  },
  {
    "doc_id": "D3",
    "title": "Independent Pilot Review \u2014 RetailCo",
    "text": "RetailCo apparel DC reported 19% pick-rate improvement after 10 weeks with MoveBot, below the vendor's 35% claim. Integration costs added ~40% to year-one spend. RetailCo notes downtime during software updates averaged 3 hours monthly."
  },
  {
    "doc_id": "D4",
    "title": "Safety Advisory \u2014 AMR Collisions",
    "text": "A regional safety board logged a 12% increase in reported AMR near-miss events YoY across all vendors. Advisories recommend denser lidar coverage in mixed pedestrian aisles. No MoveBot-specific recall is mentioned."
  },
  {
    "doc_id": "D5",
    "title": "Competitor Note \u2014 PackRunner",
    "text": "PackRunner averages $41k list and advertises 25% pick-rate gains. Independent data is scarce. PackRunner reports 200+ deployments. PackRunner lacks grocery reference customers in public materials."
  }
]

RESEARCH_QUERY = (
    "Should we run a 12-week MoveBot pilot in our apparel DC, and what claims should "
    "we treat skeptically in the vendor pitch?"
)


## Coding Section


In [ ]:
def synthesize_research_brief(query: str, sources: list) -> dict:
    """Synthesize a decision brief with claim-level evidence status.

    Args:
        query: Research / decision question.
        sources: list of doc dicts (doc_id, title, text).

    Returns:
        dict matching Functional Requirements.
    """
    # TODO: Implement your compound AI system here.
    raise NotImplementedError("Implement synthesize_research_brief")


## Example Input / Output

For `RESEARCH_QUERY` + `SOURCES`, expect the **35% pick-rate** claim to be contested by RetailCo's **19%**, integration cost uplift called out, safety note framed as industry-wide not MoveBot-specific, and a recommendation that is likely `pilot` or `needs_more_data` with clear rationale — depending on your judgment from the sources.


## Run Your Solution


In [ ]:
# result = synthesize_research_brief(RESEARCH_QUERY, SOURCES)
# print(json.dumps(result, indent=2))


## Time Limit

**Time limit: 45 minutes. No AI coding assistance.**
